In [2]:
import time
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.model_selection import KFold
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from tabulate import tabulate

# Doc bo du lieu
df = pd.read_csv("ThucHanhSVM.csv", skipinitialspace=True)
print("Du lieu sau khi doc")
print(df)

Du lieu sau khi doc
      A     B    C     D    E     F    G Class
0   9.0  15.0  3.7   2.9  1.2   5.9  4.0     P
1   3.5   4.5  3.1   2.5  2.8  10.0  6.7     N
2   4.1   4.9  2.1   3.7  2.5  10.0  3.3     P
3   4.6   3.5  2.5   2.2  2.7  11.0  7.2     P
4   5.6   1.6  4.1   3.9  4.7   5.6  2.9     N
5   5.7   3.9  2.6   5.8  2.3   4.6  3.6     P
6   8.9   2.5  2.4   3.5  1.6   9.5  2.8     P
7   2.9   3.2  2.9   3.9  3.9  10.6  4.0     P
8   2.2   2.2  3.9  13.4  2.2  11.3  2.9     N
9   6.3   4.1  2.4  12.1  1.8   2.7  4.0     P
10  2.3   3.1  3.5   2.2  5.2   5.8  3.0     N
11  7.8   2.7  3.4   2.7  5.1   3.4  2.6     P


In [3]:
# Tim cac cot chua gia tri so
num_cols = df.select_dtypes(include=['number']).columns.tolist()
print(num_cols)


['A', 'B', 'C', 'D', 'E', 'F', 'G']


In [4]:
# Tim cac cot chua gia tri ky tu
sym_cols = df.select_dtypes(include=['object', 'string', 'category']).columns.tolist()
print(sym_cols)


['Class']


In [5]:
# Chuyen cac cot chua gia tri ky tu ve dang so
df[sym_cols] = df[sym_cols].apply(lambda col: LabelEncoder().fit_transform(col))
df[sym_cols] = df[sym_cols].astype('int')
print("Du lieu sau khi chuyen doi cac cot ky tu sang so:")
print(df)


Du lieu sau khi chuyen doi cac cot ky tu sang so:
      A     B    C     D    E     F    G  Class
0   9.0  15.0  3.7   2.9  1.2   5.9  4.0      1
1   3.5   4.5  3.1   2.5  2.8  10.0  6.7      0
2   4.1   4.9  2.1   3.7  2.5  10.0  3.3      1
3   4.6   3.5  2.5   2.2  2.7  11.0  7.2      1
4   5.6   1.6  4.1   3.9  4.7   5.6  2.9      0
5   5.7   3.9  2.6   5.8  2.3   4.6  3.6      1
6   8.9   2.5  2.4   3.5  1.6   9.5  2.8      1
7   2.9   3.2  2.9   3.9  3.9  10.6  4.0      1
8   2.2   2.2  3.9  13.4  2.2  11.3  2.9      0
9   6.3   4.1  2.4  12.1  1.8   2.7  4.0      1
10  2.3   3.1  3.5   2.2  5.2   5.8  3.0      0
11  7.8   2.7  3.4   2.7  5.1   3.4  2.6      1


In [6]:
# Chuan hoa min-max cac cot dang so
df[num_cols] = MinMaxScaler().fit_transform(df[num_cols])
print("Du lieu sau khi chuan hoa min-max cac cot du lieu dang so:")
print(df)

Du lieu sau khi chuan hoa min-max cac cot du lieu dang so:
           A         B     C         D      E         F         G  Class
0   1.000000  1.000000  0.80  0.062500  0.000  0.372093  0.304348      1
1   0.191176  0.216418  0.50  0.026786  0.400  0.848837  0.891304      0
2   0.279412  0.246269  0.00  0.133929  0.325  0.848837  0.152174      1
3   0.352941  0.141791  0.20  0.000000  0.375  0.965116  1.000000      1
4   0.500000  0.000000  1.00  0.151786  0.875  0.337209  0.065217      0
5   0.514706  0.171642  0.25  0.321429  0.275  0.220930  0.217391      1
6   0.985294  0.067164  0.15  0.116071  0.100  0.790698  0.043478      1
7   0.102941  0.119403  0.40  0.151786  0.675  0.918605  0.304348      1
8   0.000000  0.044776  0.90  1.000000  0.250  1.000000  0.065217      0
9   0.602941  0.186567  0.15  0.883929  0.150  0.000000  0.304348      1
10  0.014706  0.111940  0.70  0.000000  1.000  0.360465  0.086957      0
11  0.823529  0.082090  0.65  0.044643  0.975  0.081395  0.000000

In [8]:
#Tach du lieu thanh df_x va df_y
#df_x bieu dien dac trung du lieu dau vao (ma tran)
df_x = df[num_cols].values
#df_y bieu dien nhan cua du lieu (vector)
df_y = df[sym_cols[-1]].values
print(df_x.shape)
print(df_y.shape)

(12, 7)
(12,)


In [11]:
#Chuan bi du lieu huan luyen
#Su dung phuong phap kiem dinh cheo k-fold voi k = 3
kf = KFold(n_splits=3, shuffle=False)

#Mang chua du lieu
x_train = []
x_test = []
y_train = []
y_test = []

i = 1
for train_index, test_index in kf.split(df_x):
    #Chia du lieu thanh training va testing
    x_train_sub, x_test_sub = df_x[train_index], df_x[test_index]
    y_train_sub, y_test_sub = df_y[train_index], df_y[test_index]

    #Append vao cac mang
    x_train.append(x_train_sub)
    y_train.append(y_train_sub)

    x_test.append(x_test_sub)
    y_test.append(y_test_sub)

    print(f"------Fold {i}------")
    print("Tap huan luyen: ")
    display(df.iloc[train_index])
    print("Tap kiem thu: ")
    display(df.iloc[test_index])
    i = i+1


------Fold 1------
Tap huan luyen: 


,A,B,C,D,E,F,G,Class
4,0.500000,0.000000,1.00,0.151786,0.875,0.337209,0.065217,0
5,0.514706,0.171642,0.25,0.321429,0.275,0.220930,0.217391,1
6,0.985294,0.067164,0.15,0.116071,0.100,0.790698,0.043478,1
7,0.102941,0.119403,0.40,0.151786,0.675,0.918605,0.304348,1
8,0.000000,0.044776,0.90,1.000000,0.250,1.000000,0.065217,0
9,0.602941,0.186567,0.15,0.883929,0.150,0.000000,0.304348,1
10,0.014706,0.111940,0.70,0.000000,1.000,0.360465,0.086957,0
11,0.823529,0.082090,0.65,0.044643,0.975,0.081395,0.000000,1


Tap kiem thu: 


,A,B,C,D,E,F,G,Class
0,1.000000,1.000000,0.8,0.062500,0.000,0.372093,0.304348,1
1,0.191176,0.216418,0.5,0.026786,0.400,0.848837,0.891304,0
2,0.279412,0.246269,0.0,0.133929,0.325,0.848837,0.152174,1
3,0.352941,0.141791,0.2,0.000000,0.375,0.965116,1.000000,1


------Fold 2------
Tap huan luyen: 


,A,B,C,D,E,F,G,Class
0,1.000000,1.000000,0.80,0.062500,0.000,0.372093,0.304348,1
1,0.191176,0.216418,0.50,0.026786,0.400,0.848837,0.891304,0
2,0.279412,0.246269,0.00,0.133929,0.325,0.848837,0.152174,1
3,0.352941,0.141791,0.20,0.000000,0.375,0.965116,1.000000,1
8,0.000000,0.044776,0.90,1.000000,0.250,1.000000,0.065217,0
9,0.602941,0.186567,0.15,0.883929,0.150,0.000000,0.304348,1
10,0.014706,0.111940,0.70,0.000000,1.000,0.360465,0.086957,0
11,0.823529,0.082090,0.65,0.044643,0.975,0.081395,0.000000,1


... (2 outputs left out)

------Fold 3------
Tap huan luyen: 


,A,B,C,D,E,F,G,Class
0,1.000000,1.000000,0.80,0.062500,0.000,0.372093,0.304348,1
1,0.191176,0.216418,0.50,0.026786,0.400,0.848837,0.891304,0
2,0.279412,0.246269,0.00,0.133929,0.325,0.848837,0.152174,1
3,0.352941,0.141791,0.20,0.000000,0.375,0.965116,1.000000,1
4,0.500000,0.000000,1.00,0.151786,0.875,0.337209,0.065217,0
5,0.514706,0.171642,0.25,0.321429,0.275,0.220930,0.217391,1
6,0.985294,0.067164,0.15,0.116071,0.100,0.790698,0.043478,1
7,0.102941,0.119403,0.40,0.151786,0.675,0.918605,0.304348,1


Tap kiem thu: 


,A,B,C,D,E,F,G,Class
8,0.000000,0.044776,0.90,1.000000,0.250,1.000000,0.065217,0
9,0.602941,0.186567,0.15,0.883929,0.150,0.000000,0.304348,1
10,0.014706,0.111940,0.70,0.000000,1.000,0.360465,0.086957,0
11,0.823529,0.082090,0.65,0.044643,0.975,0.081395,0.000000,1


In [ ]:
#Su dung du lieu vua chuan bi, thuc hien
#Xay dung mo hinh phan lop SVM voi kernel la ham da thuc
#Danh gia cac chi so accuracy, precision, recall, time tren tung fold
#Dua ra ket qua danh gia trung binh tren cac fold

In [14]:
#Mang chua cac ket qua
times = []
accuracies = []
precisions = []
recalls = []

i = 1
for i in range(3):
    #Khoi tao mo hinh
    svm_clf = SVC(kernel = 'poly')
    start_time = time.time()

    #Huan luyen mo hinh SVM
    svm_clf.fit(x_train[i], y_train[i])

    #Du doan mo hinh
    y_pred = svm_clf.predict(x_test[i])
    finish_time = time.time() - start_time

    #Danh gia mo hinh
    accuracy = accuracy_score(y_test[i], y_pred)
    precision = precision_score(y_test[i], y_pred)
    recall = recall_score(y_test[i], y_pred)

    #Append vao mang kq
    times.append(finish_time)
    accuracies.append(accuracy)
    precisions.append(precision)
    recalls.append(recall)

    #In ra ket qua tung fold
    #Tao du lieu bang
    data = [
        ["Chi so", "Gia tri"],
        ["Training time (s)", f"{finish_time:.4f}s"],
        ["Accuracy (%)", f"{accuracy * 100:.2f}%"],
        ["Precision (%)", f"{precision * 100:.2f}%"],
        ["Recall (%)", f"{recall * 100:.2f}%"]
    ]

    print(f"------Fold {i+1}------")
    #In bang ket qua
    print(tabulate(data, tablefmt="grid", headers="firstrow"))
    i = i+1

#Tinh cac gia tri trung binh
mean_times = np.mean(times)
std_times = np.std(times)

mean_arrcuracy = np.mean(accuracies)
std_arraccuracy = np.std(accuracies)

mean_precision = np.mean(precisions)
std_precision = np.std(precisions)

mean_recall = np.mean(recalls)
std_recall = np.std(recalls)

#In ra kq trung binh
data = [
    ["Chi so", "Mean (%)", "Std (%)"],
    ["Times", f"{mean_times:.4f}s", f"{std_times:.4f}s"],
    ["Accuracy", f"{mean_arrcuracy * 100:.2f}%", f"{std_arraccuracy * 100:.2f}%"],
    ["Precision", f"{mean_precision * 100:.2f}%", f"{std_precision * 100:.2f}%"],
    ["Recall", f"{mean_recall * 100:.2f}%", f"{std_recall * 100:.2f}%"]
]

#In bang
print("Bang gia tri trung binh ket qua tren cac fold: ")
print(tabulate(data, tablefmt="grid", headers="firstrow"))


------Fold 1------
+-------------------+-----------+
| Chi so            | Gia tri   |
+===================+===========+
| Training time (s) | 0.0019s   |
+-------------------+-----------+
| Accuracy (%)      | 75.00%    |
+-------------------+-----------+
| Precision (%)     | 75.00%    |
+-------------------+-----------+
| Recall (%)        | 100.00%   |
+-------------------+-----------+
------Fold 2------
+-------------------+-----------+
| Chi so            | Gia tri   |
+===================+===========+
| Training time (s) | 0.0010s   |
+-------------------+-----------+
| Accuracy (%)      | 75.00%    |
+-------------------+-----------+
| Precision (%)     | 100.00%   |
+-------------------+-----------+
| Recall (%)        | 66.67%    |
+-------------------+-----------+
------Fold 3------
+-------------------+-----------+
| Chi so            | Gia tri   |
+===================+===========+
| Training time (s) | 0.0009s   |
+-------------------+-----------+
| Accuracy (%)      | 50.